In [33]:
from Bio.PDB import PDBParser
import tempfile
from pyrosetta import Pose, pose_from_pdb, init, dump_pdb
from math import exp
from random import random
from utils import random_mutation, relax_structure

In [2]:
init()

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python313.Release 2026.29+release.80a0635615099e1b918474a63acba7b1de6fd107 2026-07-14T16:24:11] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.conda.ubuntu.cxx11th

In [15]:
class InteractionError(Exception):
    pass

def interaction(pose, chain1, chain2, pos1, pos2, interaction):
    """Calculate the shortest potential hydrogen-bond distance between two residues.

    :param pose: PyRosetta Pose containing the residues
    :param chain1: PDB chain ID for the first residue
    :param chain2: PDB chain ID for the second residue
    :param pos1: PDB residue position for the first residue
    :param pos2: PDB residue position for the second residue
    :return: shortest distance between potential hydrogen-bonding atoms in Angstroms
    """
    
    # Create a Biopython PDB parser
    parser = PDBParser(QUIET=True)

    # Create a temporary PDB file to store the PyRosetta pose
    with tempfile.NamedTemporaryFile(suffix=".pdb") as tmp:
        # Write the pose to the temporary PDB file
        pose.dump_pdb(tmp.name)

        # Parse the temporary PDB file into a Biopython Structure object
        structure = parser.get_structure(
            "pose",
            tmp.name
        )

    # Get the first model from the structure
    model = structure[0]

    # Retrieve the two chains being analyzed
    chainA = model[chain1]
    chainB = model[chain2]

    # Retrieve the residues at the specified PDB residue positions
    res1 = chainA[pos1]
    res2 = chainB[pos2]

    # Get the three-letter amino acid codes for each residue
    aa1 = res1.get_resname()
    aa2 = res2.get_resname()

    # Get the hydrogen-bonding atoms for each residue
    # The returned atom lists correspond to res1 and res2 respectively
    
    if interaction == 'Hydrogen Bond':
        atoms1, atoms2 = hydrogen_bond_atoms(aa1, aa2)
    elif interaction == 'Salt Bridge':
        atoms1, atoms2 = salt_bridge_atoms(aa1, aa2)
    else:
        raise InteractionError('Interaction must be "Hydrogen Bond" or "Salt Bridge"')

    # Store all possible donor/acceptor atom pair distances
    distance = []

    # Compare every possible hydrogen-bonding atom in res1
    # against every possible hydrogen-bonding atom in res2
    for atom1 in atoms1:
        for atom2 in atoms2:

            # Calculate the distance between the two atoms in Angstroms
            distance.append(res1[atom1] - res2[atom2])

    # Return the shortest distance between any possible
    # hydrogen-bonding atom pair
    return min(distance)

In [16]:
class HydrogenBondError(Exception):
    pass

def hydrogen_bond_atoms(aa1, aa2):
    """Identify potential hydrogen-bonding atoms between two amino acids.

    :param aa1: Three-letter amino acid code for the first residue
    :param aa2: Three-letter amino acid code for the second residue
    :return: lists of hydrogen-bond donor/acceptor atoms corresponding to aa1 and aa2
    """

    # Define atoms that can donate a hydrogen bond for each amino acid
    hbond_donors = {
        "ARG": ["NE", "NH1", "NH2"],
        "ASN": ["ND2"],
        "CYS": ["SG"],
        "GLN": ["NE2"],
        "HIS": ["ND1", "NE2"],
        "LYS": ["NZ"],
        "SER": ["OG"],
        "THR": ["OG1"],
        "TRP": ["NE1"],
        "TYR": ["OH"]
    }

    # Define atoms that can accept a hydrogen bond for each amino acid
    hbond_acceptors = {
        "ASN": ["OD1"],
        "ASP": ["OD1", "OD2"],
        "CYS": ["SG"],
        "GLN": ["OE1"],
        "GLU": ["OE1", "OE2"],
        "HIS": ["ND1", "NE2"],
        "MET": ["SD"],
        "SER": ["OG"],
        "THR": ["OG1"],
        "TYR": ["OH"]
    }

    # Check whether aa1 can donate and aa2 can accept a hydrogen bond
    if (aa1 in hbond_donors and aa2 in hbond_acceptors):
        return hbond_donors[aa1], hbond_acceptors[aa2]

    # Check whether aa2 can donate and aa1 can accept a hydrogen bond
    elif aa2 in hbond_donors and aa1 in hbond_acceptors:
        return hbond_acceptors[aa1], hbond_donors[aa2]

    # Raise an error if neither residue can form a potential hydrogen bond
    else:
        raise HydrogenBondError(
            f'{aa1} and {aa2} cannot make a hydrogen bond'
        )

In [19]:
interaction(pose_from_pdb('pdb_files/7K18_relaxed.pdb'), 'A', 'B', 1612, 15, 'Salt Bridge')

core.import_pose.import_pose: File 'pdb_files/7K18_relaxed.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue ALA:CtermProteinFull 100
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149


np.float32(4.991876)

In [13]:
class SaltBridgeError(Exception):
    pass


def salt_bridge_atoms(aa1, aa2):
    """Identify potential salt-bridge atoms between two amino acids.

    :param aa1: Three-letter amino acid code for the first residue
    :param aa2: Three-letter amino acid code for the second residue
    :return: lists of salt-bridge atoms corresponding to aa1 and aa2
    """

    # Define negatively charged atoms that can participate in salt bridges
    salt_bridge_negative = {
        "ASP": ["OD1", "OD2"],
        "GLU": ["OE1", "OE2"]
    }

    # Define positively charged atoms that can participate in salt bridges
    salt_bridge_positive = {
        "ARG": ["NE", "NH1", "NH2"],
        "LYS": ["NZ"],
        "HIS": ["ND1", "NE2"]
    }

    # Check whether aa1 is positively charged and aa2 is negatively charged
    if (aa1 in salt_bridge_positive and aa2 in salt_bridge_negative):
        return salt_bridge_positive[aa1], salt_bridge_negative[aa2]

    # Check whether aa1 is negatively charged and aa2 is positively charged
    elif (aa1 in salt_bridge_negative and aa2 in salt_bridge_positive):
        return salt_bridge_negative[aa1], salt_bridge_positive[aa2]

    # Raise an error if the residues cannot form a salt bridge
    else:
        raise SaltBridgeError(
            f'{aa1} and {aa2} cannot make a salt bridge'
        )

In [32]:
class InteractionError(Exception):
    pass

class SaltBridgeError(Exception):
    pass

class HydrogenBondError(Exception):
    pass

class AffinityOptimizer():

    def __init__(self, pdb, scoring_function, pos_to_mutate, temp = 10, cooling_rate = 0.95):
        self.pdb = pdb
        self.pose = pose_from_pdb(pdb)
        self.pos_to_mutate = pos_to_mutate
        self.scoring_function = scoring_function
        self.temp = temp
        self.cooling_rate = cooling_rate

        if scoring_function == 'Distance':
            print('Must insert Interactions to optimize')
            print('Example .insert_interaction(1612, 43)')
            self.func = self.distance
            self.distances = []
            self.chains = []
            self.positions = []
            self.interactions = []

    def interaction_distance(self, chain1, chain2, pos1, pos2, interaction):
        """Calculate the shortest potential hydrogen-bond distance between two residues.

        :param pose: PyRosetta Pose containing the residues
        :param chain1: PDB chain ID for the first residue
        :param chain2: PDB chain ID for the second residue
        :param pos1: PDB residue position for the first residue
        :param pos2: PDB residue position for the second residue
        :return: shortest distance between potential hydrogen-bonding atoms in Angstroms
        """
        
        # Create a Biopython PDB parser
        parser = PDBParser(QUIET=True)

        structure = self.get_structure()

        # Get the first model from the structure
        model = structure[0]

        # Retrieve the two chains being analyzed
        chainA = model[chain1]
        chainB = model[chain2]

        # Retrieve the residues at the specified PDB residue positions
        res1 = chainA[pos1]
        res2 = chainB[pos2]

        # Get the three-letter amino acid codes for each residue
        aa1 = res1.get_resname()
        aa2 = res2.get_resname()

        # Get the hydrogen-bonding atoms for each residue
        # The returned atom lists correspond to res1 and res2 respectively
        
        if interaction == 'Hydrogen Bond':
            atoms1, atoms2 = self.hydrogen_bond_atoms(aa1, aa2)
        elif interaction == 'Salt Bridge':
            atoms1, atoms2 = self.salt_bridge_atoms(aa1, aa2)
        else:
            raise InteractionError('Interaction must be "Hydrogen Bond" or "Salt Bridge"')

        # Store all possible donor/acceptor atom pair distances
        distance = []

        # Compare every possible hydrogen-bonding atom in res1
        # against every possible hydrogen-bonding atom in res2
        for atom1 in atoms1:
            for atom2 in atoms2:

                # Calculate the distance between the two atoms in Angstroms
                distance.append(res1[atom1] - res2[atom2])

        # Return the shortest distance between any possible
        # hydrogen-bonding atom pair
        return float(min(distance))

    def salt_bridge_atoms(self, aa1, aa2):
        """Identify potential salt-bridge atoms between two amino acids.

        :param aa1: Three-letter amino acid code for the first residue
        :param aa2: Three-letter amino acid code for the second residue
        :return: lists of salt-bridge atoms corresponding to aa1 and aa2
        """

        # Define negatively charged atoms that can participate in salt bridges
        salt_bridge_negative = {
            "ASP": ["OD1", "OD2"],
            "GLU": ["OE1", "OE2"]
        }

        # Define positively charged atoms that can participate in salt bridges
        salt_bridge_positive = {
            "ARG": ["NE", "NH1", "NH2"],
            "LYS": ["NZ"],
            "HIS": ["ND1", "NE2"]
        }

        # Check whether aa1 is positively charged and aa2 is negatively charged
        if (aa1 in salt_bridge_positive and aa2 in salt_bridge_negative):
            return salt_bridge_positive[aa1], salt_bridge_negative[aa2]

        # Check whether aa1 is negatively charged and aa2 is positively charged
        elif (aa1 in salt_bridge_negative and aa2 in salt_bridge_positive):
            return salt_bridge_negative[aa1], salt_bridge_positive[aa2]

        # Raise an error if the residues cannot form a salt bridge
        else:
            raise SaltBridgeError(
                f'{aa1} and {aa2} cannot make a salt bridge'
            )

    def hydrogen_bond_atoms(self, aa1, aa2):
        """Identify potential hydrogen-bonding atoms between two amino acids.

        :param aa1: Three-letter amino acid code for the first residue
        :param aa2: Three-letter amino acid code for the second residue
        :return: lists of hydrogen-bond donor/acceptor atoms corresponding to aa1 and aa2
        """

        # Define atoms that can donate a hydrogen bond for each amino acid
        hbond_donors = {
            "ARG": ["NE", "NH1", "NH2"],
            "ASN": ["ND2"],
            "CYS": ["SG"],
            "GLN": ["NE2"],
            "HIS": ["ND1", "NE2"],
            "LYS": ["NZ"],
            "SER": ["OG"],
            "THR": ["OG1"],
            "TRP": ["NE1"],
            "TYR": ["OH"]
        }

        # Define atoms that can accept a hydrogen bond for each amino acid
        hbond_acceptors = {
            "ASN": ["OD1"],
            "ASP": ["OD1", "OD2"],
            "CYS": ["SG"],
            "GLN": ["OE1"],
            "GLU": ["OE1", "OE2"],
            "HIS": ["ND1", "NE2"],
            "MET": ["SD"],
            "SER": ["OG"],
            "THR": ["OG1"],
            "TYR": ["OH"]
        }

        # Check whether aa1 can donate and aa2 can accept a hydrogen bond
        if (aa1 in hbond_donors and aa2 in hbond_acceptors):
            return hbond_donors[aa1], hbond_acceptors[aa2]

        # Check whether aa2 can donate and aa1 can accept a hydrogen bond
        elif aa2 in hbond_donors and aa1 in hbond_acceptors:
            return hbond_acceptors[aa1], hbond_donors[aa2]

        # Raise an error if neither residue can form a potential hydrogen bond
        else:
            raise HydrogenBondError(
                f'{aa1} and {aa2} cannot make a hydrogen bond'
            )

    def insert_interaction(self, chain1, chain2, pos1, pos2, interaction):
        self.distances.append(self.interaction_distance(chain1, chain2, pos1, pos2, interaction))
        self.chains.append([chain1, chain2])
        self.positions.append([pos1, pos2])
        self.interactions.append(interaction)

    def distance_score(self, pose_distances):
        sum_dis = 0

        for dis in pose_distances:
            sum_dis += dis - 2.8

        return sum_dis

    def find_distances(self, pose=None):
        if pose is None:
            pose = self.pose
        dis = []

        for i, (dbl_chain, dbl_pos) in enumerate(zip(self.chains, self.positions)):
            dis.append(self.interaction_distance(dbl_chain[0], dbl_chain[1], dbl_pos[0], dbl_pos[1], self.interactions[i]))

        return dis

    def distance(self, new_pose):

        new_distances = self.find_distances(new_pose)
        old_distances = self.distances

        new_score = self.distance_score(new_distances)
        old_score = self.distance_score(old_distances)

        delta_score = new_score - old_score

        if delta_score < 0:
            self.pose = new_pose
            self.distances = new_distances

        else:
            prob = exp(-delta_score / self.temp)

            if prob > random():
                self.pose = new_pose
                self.distance = new_distances
        self.temp -= self.cooling_rate

    def run(self):

        for i in range(50):
            mutant_pose = random_mutation(self.pose, self.pos_to_mutate)
            relaxed_mutant = relax_structure(mutant_pose)
            self.func(relaxed_mutant)

        
            
                



    def view_interactions(self):
        structure = self.get_structure()
        model = structure[0]
        print("chains:", self.chains)
        print("positions:", self.positions)

        for num, (chain, pos) in enumerate(zip(self.chains, self.positions)):
            aa1 = model[chain[0]][pos[0]].get_resname()
            aa2 = model[chain[1]][pos[1]].get_resname()
            print(f'Interaction {num + 1}')
            print(f'AA: {aa1}, chain: {chain[0]}, pos: {pos[0]}')
            print(f'AA: {aa2}, chain: {chain[1]}, pos: {pos[1]}')
            print(f'Distance: {self.distances[num]}')


    def get_structure(self):
        # Create a Biopython PDB parser
        parser = PDBParser(QUIET=True)

        # Create a temporary PDB file to store the PyRosetta pose
        with tempfile.NamedTemporaryFile(suffix=".pdb") as tmp:
            # Write the pose to the temporary PDB file
            self.pose.dump_pdb(tmp.name)

            # Parse the temporary PDB file into a Biopython Structure object
            structure = parser.get_structure(
                "pose",
                tmp.name
            )
        return structure


        
        
        

In [ ]:
aa_mutate = [104, 107, 108, 109, 110, 113, 114, 117, 118, 119, 120, 121, 
             131, 141, 142, 151, 154, 155, 156, 157, 159, 160, 161, 162, 
             166, 167]

opt = AffinityOptimizer('pdb_files/7K18_relaxed.pdb', 'Distance', aa_mutate)
opt.insert_interaction('A', 'B', 1612, 43, 'Hydrogen Bond')
opt.insert_interaction('A', 'B', 1617, 64, 'Hydrogen Bond')
opt.insert_interaction('A', 'B', 1612, 15, 'Salt Bridge')
opt.insert_interaction('A', 'B', 1612, 65, 'Hydrogen Bond')
opt.view_interactions()
opt.find_distances()
opt.run()
dump_pdb(opt.pose, 'exp.pdb')

opt.view_interactions()

core.import_pose.import_pose: File 'pdb_files/7K18_relaxed.pdb' automatically determined to be of type PDB from contents.


core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue ALA:CtermProteinFull 100
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149
Must insert Interactions to optimize
Example .insert_interaction(1612, 43)
chains: [['A', 'B'], ['A', 'B'], ['A', 'B'], ['A', 'B']]
positions: [[1612, 43], [1617, 64], [1612, 15], [1612, 65]]
Interaction 1
AA: ASP, chain: A, pos: 1612
AA: HIS, chain: B, pos: 43
Distance: 2.6666457653045654
Interaction 2
AA: TYR, chain: A, pos: 1617
AA: LYS, chain: B, pos: 64
Distance: 3.0455009937286377
Interaction 3
AA: ASP, chain: A, pos: 1612
AA: HIS, chain: B, pos: 15
Distance: 4.991876125335693
Interaction 4
AA: ASP, chain: A, pos: 1612
AA: CYS, chain: B, pos: 65
Distance: 4.252884387969971

-----------------

chains: [['A', 'B'], ['A', 'B'], ['A', 'B'], ['A', 'B']]
positions: [[1612, 43], [1617, 64], [1612, 15], [1612, 65]]
Interaction 1
AA: ASP, chain: A, pos: 1612
AA: HIS, chain: B, pos: 43
Distance: 3.0663135051727295
Interaction 2
AA: TYR, chain: A, pos: 1617
AA: LYS, chain: B, pos: 64
Distance: 3.0455009937286377
Interaction 3
AA: ASP, chain: A, pos: 1612
AA: HIS, chain: B, pos: 15
Distance: 4.555408477783203
Interaction 4
AA: ASP, chain: A, pos: 1612
AA: CYS, chain: B, pos: 65
Distance: 4.252884387969971
